# L7.7 · SmolVLA: el entrenamiento por dentro

| Notebook | Qué abre |
|---|---|
| L7.1 | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| L7.2 | La imagen: de la foto a 64 tokens |
| L7.3 | La fila del VLM: palabras, estado y quién puede mirar a quién |
| L7.4 | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| L7.5 | El experto de acciones: la brújula de L6 con ojos |
| L7.6 | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| **L7.7** | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import inspect

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.modeling_smolvla import make_att_2d_masks

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR
from toolkit.policy.smolvla_g1_base import load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
FPS = 30

TRAINED = CHECKPOINTS_DIR / "smolvla_g1_v1" / "checkpoints" / "last" / "pretrained_model"
train = LeRobotDataset("local/pick_mameluco_act_v1_train", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1_train",
                       delta_timestamps={"action": [i / FPS for i in range(50)]})
policy, preprocess, _ = load_g1_policy(TRAINED, train.meta.stats, device)
model = policy.model

loader = torch.utils.data.DataLoader(train, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(0))
batch = preprocess(next(iter(loader)))
print("un lote de entrenamiento:", {k: tuple(v.shape) for k, v in batch.items() if hasattr(v, "shape")})

## 1. El problema

El experto de L7.5 aprendió a ser una brújula. ¿Cómo? Con la misma pérdida de
cuatro líneas de L6.2, sólo que el "punto" es un bloque de 50 × 32 y la
brújula mira la escena. Acá desarmamos **un paso de entrenamiento** con un
lote real de 8 instantes, y después miramos tres problemas que tiene LeRobot
0.4.1 al entrenar con nuestro robot.

## 2. Un paso, a mano

Las cuatro líneas de L6.2, con la convención de LeRobot (`t = 1` es ruido):

1. Sorteo ruido del tamaño del bloque, y un momento `t` para cada instante del
   lote.
2. El punto del camino: `x_t = t · ruido + (1 - t) · bloque`.
3. La velocidad correcta de esa pareja: `u = ruido - bloque`.
4. Le pido al modelo la velocidad en `(x_t, t)`, mirando la escena, y la comparo
   con `u` con error cuadrático.

Lo escribimos con las piezas del modelo y verificamos que da exactamente lo
mismo que su `forward`:

In [ ]:
images, img_masks = policy.prepare_images(batch)
state = policy.prepare_state(batch)
actions = policy.prepare_action(batch)  # (8, 50, 32): las 26 articulaciones + 6 ceros
lang_tokens, lang_masks = batch["observation.language.tokens"], batch["observation.language.attention_mask"]

torch.manual_seed(0)
noise = torch.randn_like(actions)                                 # 1. ruido...
t = model.sample_time(actions.shape[0], device)                   #    ...y un momento por instante
x_t = t[:, None, None] * noise + (1 - t[:, None, None]) * actions  # 2. el punto del camino
u = noise - actions                                               # 3. la velocidad correcta

with torch.no_grad():                                             # 4. la velocidad del modelo
    prefix_embs, prefix_pad, prefix_att = model.embed_prefix(images, img_masks, lang_tokens, lang_masks, state=state)
    suffix_embs, suffix_pad, suffix_att = model.embed_suffix(x_t, t)
    pad = torch.cat([prefix_pad, suffix_pad], dim=1)
    att = torch.cat([prefix_att, suffix_att], dim=1)
    (_, out), _ = model.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(pad, att), position_ids=torch.cumsum(pad, dim=1) - 1,
        past_key_values=None, inputs_embeds=[prefix_embs, suffix_embs], use_cache=False, fill_kv_cache=False)
    v = model.action_out_proj(out[:, -model.config.chunk_size:].float())
    losses = (v - u) ** 2

    reference = model.forward(images, img_masks, lang_tokens, lang_masks, state, actions, noise=noise, time=t)
print("igual al forward del modelo:", torch.allclose(losses, reference, atol=1e-5), "· pérdida:", round(losses.mean().item(), 4))

Una diferencia con L6.2: en el juguete, el momento se sorteaba parejo entre 0
y 1. Acá sale de una distribución Beta que **favorece los momentos ruidosos**,
igual que en π0:

In [ ]:
samples = model.sample_time(100_000, device).cpu().numpy()
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(samples, bins=50, density=True, color="tab:red", alpha=0.7, label="lo que sortea SmolVLA")
ax.axhline(1, color="gray", ls="--", label="parejo (L6.2)")
ax.set_xlabel("momento t (1 = ruido, 0 = bloque terminado)")
ax.set_ylabel("densidad")
ax.legend()
plt.tight_layout()
print(f"momentos con t > 0,5: {(samples > 0.5).mean():.0%}")

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## 3. Dónde está el error

La pérdida es un número por cada acción del bloque y cada uno de los 32
números. Promediando el lote:

In [ ]:
per_cell = losses.mean(0).cpu().numpy()  # (50 acciones, 32 números)
fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})
im = axes[0].imshow(per_cell.T, aspect="auto", cmap="viridis")
axes[0].set_xlabel("acción del bloque")
axes[0].set_ylabel("número (0-25 articulaciones, 26-31 relleno)")
axes[0].axhline(25.5, color="white", lw=1)
fig.colorbar(im, ax=axes[0])
axes[1].bar(range(32), per_cell.mean(0), color=["tab:red"] * 26 + ["lightgray"] * 6)
axes[1].set_xlabel("número")
axes[1].set_title("pérdida promedio por número")
plt.tight_layout()

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## 4. Tres problemas de LeRobot 0.4.1

**Problema 1: el relleno de fin de episodio cuenta.** Cuando un instante está
a menos de 50 cuadros del final, al bloque le faltan acciones, y el dataset lo
completa repitiendo la última y lo marca en `action_is_pad`. El `forward` de
SmolVLA debería ignorar esas posiciones, pero busca otro nombre:

In [ ]:
source = inspect.getsource(type(policy).forward)
print([line.strip() for line in source.splitlines() if "pad" in line and "batch" in line])
print("¿el lote trae 'action_is_pad'?", "action_is_pad" in batch, "· ¿y 'actions_id_pad'?", "actions_id_pad" in batch)

lengths = np.array(train.meta.episodes["dataset_to_index"]) - np.array(train.meta.episodes["dataset_from_index"])
padded = sum(sum(max(0, i + 50 - n) for i in range(n)) for n in lengths)
print(f"posiciones de relleno en todo el entrenamiento: {padded / (lengths.sum() * 50):.1%}")

__PAD1_TEXT__

**Problema 2: los 6 números de relleno entran en la pérdida.** Las
articulaciones se rellenan con ceros hasta 32 (L7.3), y la pérdida se promedia
sobre los 32. Para esos 6 la respuesta es trivial, así que el modelo los
aprende enseguida (sección 3) y el número que se reporta queda más bajo de lo
que en verdad es:

In [ ]:
real = losses[..., :26].mean().item()
reported = losses.mean().item()
print(f"pérdida reportada (32 números): {reported:.4f} · pérdida en las 26 articulaciones: {real:.4f} "
      f"→ el número reportado es {1 - reported / real:.0%} más bajo")

**Problema 3: las formas del robot preentrenado.** Ya lo vimos en `06`: al
partir de `smolvla_base`, la configuración conservaba las 6 articulaciones del
SO-100 y habría recortado nuestras acciones a 6. Lo esquivamos con una copia
local de la configuración.

Los tres están arreglados en la versión actual de LeRobot (lo verificamos en
su código). El 1 y el 2 afectan cómo aprende; el 3 rompía el resultado. Para
nuestro entrenamiento dejamos el 1 y el 2 tal cual, para que fuera el flujo
oficial sin cambios: es la referencia. Arreglar el 1 es un buen primer
experimento de una sola variable.

## 5. Qué se entrena

Hacemos el paso hacia atrás y miramos qué partes reciben gradiente:

In [ ]:
policy.train()
model.forward(images, img_masks, lang_tokens, lang_masks, state, actions, noise=noise, time=t).mean().backward()
parts = {
    "visión (SigLIP)": model.vlm_with_expert.get_vlm_model().vision_model,
    "VLM de texto": model.vlm_with_expert.get_vlm_model().text_model,
    "experto": model.vlm_with_expert.lm_expert,
    "estado → token": model.state_proj,
    "acciones ↔ tokens y momento": torch.nn.ModuleList([model.action_in_proj, model.action_out_proj,
                                                         model.action_time_mlp_in, model.action_time_mlp_out]),
}
for name, part in parts.items():
    with_grad = sum(p.numel() for p in part.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
    total = sum(p.numel() for p in part.parameters())
    print(f"  {name:<28} {with_grad:>12,} de {total:>12,} parámetros reciben gradiente".replace(",", "."))
policy.zero_grad(set_to_none=True)
policy.eval()

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Lo que nos llevamos

- Un paso de entrenamiento es la pérdida de L6.2 en 1600 dimensiones, con la
  escena de por medio. Lo reconstruimos a mano y da lo mismo que el modelo.
- El momento se sortea favoreciendo la parte ruidosa del camino.
- LeRobot 0.4.1 tiene tres problemas al entrenar con un robot distinto del
  preentrenado; esquivamos el que rompía y dejamos documentados los otros dos.
- Se entrena el experto, las proyecciones y la matriz del estado. El VLM
  queda congelado, pero el gradiente lo atraviesa para llegar a la matriz del
  estado.

## Referencias

- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025), sección
  3.1 (la pérdida) y 4.3 (detalles de entrenamiento).
- Black et al., [π0](https://arxiv.org/abs/2410.24164) (2024): la distribución
  Beta del momento.
- Código: `VLAFlowMatching.forward` y `SmolVLAPolicy.forward` en
  `lerobot/policies/smolvla/modeling_smolvla.py` (0.4.1 contra `main`).